In [ ]:
import sys
from pathlib import Path

current_path = Path.cwd()
PROJECT_ROOT = None

for p in [current_path, current_path.parent, current_path.parent.parent]:
    if (p / "rainfall_acoustic_classification").exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT:
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT)) # insert(0) garante prioridade absoluta
    print(f"Raiz do projeto adicionada ao sys.path: {PROJECT_ROOT}")
else:
    print("ERRO: Não foi possível encontrar a raiz do projeto.")

✅ Raiz do projeto adicionada ao sys.path: c:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-Rainfall\RAC-semiurban-forest-ML-2026


# Parallel Pipeline

In [2]:
%%writefile UECE_pipeline.py
import sys
import gc
from pathlib import Path
from typing import Dict, Any, List

# ====================================================================
# 1. PATH INJECTION FOR WORKERS
# Child processes must discover the project root independently.
# ====================================================================
current_path = Path.cwd()
for p in [current_path, current_path.parent, current_path.parent.parent]:
    if (p / "rainfall_acoustic_classification").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

# ====================================================================
# 2. LIBRARY IMPORTS AND CONFIGURATION
# ====================================================================
from rainfall_acoustic_classification.core import load_audio_sample
from rainfall_acoustic_classification.processing import (
    AudioAugmenter, AudioSegmenter, AcousticMetrics
)

augmenter = AudioAugmenter(sample_rate=24000, noise_prob=0.2, cutout_prob=0.3, random_state=42)
segmenter = AudioSegmenter(sample_rate=24000,segment_duration=10.0, overlap=0.5)
metrics_ext = AcousticMetrics(sample_rate=24000, fft_window_size=1024)

def audio_processing_worker(row_dict: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Worker function to execute signal processing and feature extraction 
    for a single audio file.

    This function loads the audio, applies stochastic augmentation (if flagged),
    segments the signal, and extracts acoustic metrics for each segment.

    Parameters
    ----------
    row_dict : dict
        A dictionary representing a single row from the metadata DataFrame. Must contain
        at least the 'file_path', 'split', and 'should_augment' keys.

    Returns
    -------
    list of dict
        A list of dictionaries, where each dictionary contains the extracted features
        and metadata for a specific segment of the original audio file. Returns an
        empty list if the audio loading fails.
    """
    sample = load_audio_sample(row_dict.get('file_path'), sample_rate=24000)
    if not sample: 
        return []
    
    y = sample.audio_data
    aug_log = "raw"
    
    # Apply stochastic augmentation strictly if the flag is True
    if row_dict.get('split') == 'train' and row_dict.get('should_augment', False):
        y, aug_log = augmenter.process(y)
        
    results = []
    chunks = segmenter.process(y)
    
    for idx, (chunk_array, offset) in enumerate(chunks):
        features = metrics_ext.calculate(chunk_array)
        
        segment_data = row_dict.copy()
        segment_data.update({
            'segment_idx': idx, 
            'offset_sec': offset, 
            'aug_params': aug_log
        })
        segment_data.update(features)
        results.append(segment_data)
        
    # =======================================================
    # AGGRESSIVE GARBAGE COLLECTION
    # =======================================================
    del y
    del chunks
    del sample
    gc.collect() 
    
    return results

Overwriting UECE_pipeline.py


# Processing

In [3]:
import pandas as pd
from rainfall_acoustic_classification.utils import parallel_pipe
from UECE_pipeline import audio_processing_worker
from rainfall_acoustic_classification.utils import prepare_balanced_training_set

split_file_path = PROJECT_ROOT / "data" / "splits" / "UECE"
metrics_file_path = PROJECT_ROOT / "data" / "processed" / "UECE"

## Train

In [4]:
train_file_path = split_file_path / "UECE_train.csv"

# Load and tag the split
df_train_meta = pd.read_csv(train_file_path)
df_train_meta['split'] = 'train'

# Define the rain classes according to the methodology
RAIN_CLASSES = ['light', 'moderate', 'heavy', 'violent']

# Apply the balancing function
df_train_final = prepare_balanced_training_set(
    df_train=df_train_meta, 
    rain_classes=RAIN_CLASSES, 
    random_state=42
)

print(f"\n Total tasks dispatched to workers: {len(df_train_final)} files.")


📊 Intra-Class Balancing Strategy:
   -> Majority Class: 'moderate' with N_max = 252 samples.
   -> [light] Originals: 239 | Augmented generated: +13 | Final Total: 252
   -> [moderate] Originals: 252 | Already at N_max. No augmentation applied.
   -> [heavy] Originals: 192 | Augmented generated: +60 | Final Total: 252
   -> [violent] Originals: 32 | Augmented generated: +220 | Final Total: 252

 Total tasks dispatched to workers: 1723 files.


In [5]:
# Execute the parallel pipeline
df_train_final_features = df_train_final.pipe(
    parallel_pipe, 
    worker_func=audio_processing_worker, 
    n_jobs=20, 
    desc="Extracting UECE Features"
)

Extracting UECE Features:   0%|          | 0/1723 [00:00<?, ?file/s]

In [6]:
train_metrics_file_path = metrics_file_path / "UECE_train_metrics.csv"
df_train_final_features.to_csv(train_metrics_file_path, index=False)

## Validation

In [7]:
val_file_path = split_file_path / "UECE_val.csv"

df_val_meta = pd.read_csv(val_file_path)
df_val_meta['split'] = 'val'

df_val_final_features = df_val_meta.pipe(
    parallel_pipe, 
    worker_func=audio_processing_worker, 
    n_jobs=20, 
    desc="Extraindo Features UECE"
)

Extraindo Features UECE:   0%|          | 0/179 [00:00<?, ?file/s]

In [8]:
val_metrics_file_path = metrics_file_path / "UECE_val_metrics.csv"
df_val_final_features.to_csv(val_metrics_file_path, index=False)

## Test

In [9]:
test_file_path = split_file_path / "UECE_test.csv"

df_test_meta = pd.read_csv(test_file_path)
df_test_meta['split'] = 'test'

df_test_final_features = df_test_meta.pipe(
    parallel_pipe, 
    worker_func=audio_processing_worker, 
    n_jobs=20, 
    desc="Extraindo Features UECE"
)

Extraindo Features UECE:   0%|          | 0/179 [00:00<?, ?file/s]

In [10]:
test_metrics_file_path = metrics_file_path / "UECE_test_metrics.csv"
df_test_final_features.to_csv(test_metrics_file_path, index=False)

In [11]:
count_train = df_train_final_features['category'].value_counts()
print('TRAIN:')
print(count_train)
print("Total Samples    ", len(df_train_final_features['category']), '\n')

count_val = df_val_final_features['category'].value_counts()
print('VALIDATION:')
print(count_val)
print("Total Samples    ", len(df_val_final_features['category']), '\n')

count_test = df_test_final_features['category'].value_counts()
print('TEST')
print(count_test)
print("Total Samples    ", len(df_test_final_features['category']), '\n')

total = len(df_test_final_features['category']) + len(df_val_final_features['category']) + len(df_train_final_features['category'])

print("PERCENTAGES:")
print("Train         ", round((len(df_train_final_features['category'])/total), 2))
print("Validation    ", round((len(df_val_final_features['category'])/total), 2))
print("Test          ", round((len(df_test_final_features['category'])/total), 2))

TRAIN:
category
no-rain     7865
moderate    2772
heavy       2772
violent     2772
light       2772
Name: count, dtype: int64
Total Samples     18953 

VALIDATION:
category
no-rain     990
moderate    341
light       330
heavy       264
violent      44
Name: count, dtype: int64
Total Samples     1969 

TEST
category
no-rain     979
moderate    352
light       330
heavy       264
violent      44
Name: count, dtype: int64
Total Samples     1969 

PERCENTAGES:
Train          0.83
Validation     0.09
Test           0.09


In [12]:
df_train_final_features.groupby(['category']).sample()

,file_name,timestamp,period,mm_5min,mm_hr,category,recorder,location,file_path,extension,...,wav_detail_lvl1_energy,wav_detail_lvl1_std,wav_energy_mean,roughness,tfsd,wav_detail_lvl1_var,wav_detail_lvl2_var,wav_detail_lvl3_var,wav_detail_lvl4_var,wav_detail_lvl5_var
1060,SMM08565UECE_20250701_051500_1-0_heavy.wav,2025-07-01 05:15:00,overnight,1.0,12.0,heavy,SMM08565,UECE,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,7.718197e-06,0.002778,0.003824,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10583,SMM08565UECE_20250327_074500_0-2_light.wav,2025-03-27 07:45:00,morning,0.2,2.4,light,SMM08565,UECE,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,7.183081e-07,0.000847,0.005006,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7251,SMM08565UECE_20250227_223000_0-8_moderate.wav,2025-02-27 22:30:00,night,0.8,9.6,moderate,SMM08565,UECE,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,1.774147e-02,0.133197,0.527832,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5325,SMM08565UECE_20250716_093500_0-0_no-rain.wav,2025-07-16 09:35:00,morning,0.0,0.0,no-rain,SMM08565,UECE,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,1.230882e-03,0.035084,0.079159,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17581,SMM08565UECE_20250628_203000_4-2_violent.wav,2025-06-28 20:30:00,night,4.2,50.4,violent,SMM08565,UECE,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,3.334982e-02,0.182619,0.914671,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df_val_final_features.groupby(['category']).sample()

,file_name,timestamp,period,mm_5min,mm_hr,category,recorder,location,file_path,extension,...,wav_detail_lvl1_energy,wav_detail_lvl1_std,wav_energy_mean,roughness,tfsd,wav_detail_lvl1_var,wav_detail_lvl2_var,wav_detail_lvl3_var,wav_detail_lvl4_var,wav_detail_lvl5_var
245,SMM08565UECE_20250702_021500_1-8_heavy.wav,2025-07-02 02:15:00,overnight,1.8,21.6,heavy,SMM08565,UECE,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,1.292785e-04,0.011370,0.014407,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1536,SMM08565UECE_20250730_051000.wav,2025-07-30 05:10:00,overnight,0.2,2.4,light,SMM08565,UECE,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,4.076536e-07,0.000638,0.000061,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1355,SMM08565UECE_20250304_072500_0-6_moderate.wav,2025-03-04 07:25:00,morning,0.6,7.2,moderate,SMM08565,UECE,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,4.560783e-04,0.021356,0.024172,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1821,SMM08565UECE_20250723_225000_0-0_no-rain.wav,2025-07-23 22:50:00,night,0.0,0.0,no-rain,SMM08565,UECE,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,8.770882e-05,0.009365,0.000187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1851,SMM08565UECE_20250702_015500_6-4_violent.wav,2025-07-02 01:55:00,overnight,6.4,76.8,violent,SMM08565,UECE,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,1.176199e-04,0.010845,0.025751,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df_test_final_features.groupby(['category']).sample()

,file_name,timestamp,period,mm_5min,mm_hr,category,recorder,location,file_path,extension,...,wav_detail_lvl1_energy,wav_detail_lvl1_std,wav_energy_mean,roughness,tfsd,wav_detail_lvl1_var,wav_detail_lvl2_var,wav_detail_lvl3_var,wav_detail_lvl4_var,wav_detail_lvl5_var
1266,SMM08565UECE_20250728_233000.wav,2025-07-28 23:30:00,night,1.4,16.8,heavy,SMM08565,UECE,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.001459,0.038195,0.058727,NaN,NaN,NaN,NaN,NaN,NaN,NaN
918,SMM08565UECE_20250721_010500_0-2_light.wav,2025-07-21 01:05:00,overnight,0.2,2.4,light,SMM08565,UECE,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.000026,0.005064,0.005198,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1405,SMM08565UECE_20250428_054000_0-4_moderate.wav,2025-04-28 05:40:00,overnight,0.4,4.8,moderate,SMM08565,UECE,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.000246,0.015688,0.010778,NaN,NaN,NaN,NaN,NaN,NaN,NaN
108,SMM08565UECE_20250511_215500_0-0_no-rain.wav,2025-05-11 21:55:00,night,0.0,0.0,no-rain,SMM08565,UECE,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.000303,0.017413,0.000187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1755,SMM08565UECE_20250406_212000_5-2_violent.wav,2025-04-06 21:20:00,night,5.2,62.4,violent,SMM08565,UECE,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.000536,0.023161,0.084041,NaN,NaN,NaN,NaN,NaN,NaN,NaN
